Oleksandr Reshetnikov (2409418)


# Introduction

### Skin Lesion Classification on ISIC 2019

Module: CM4126 Computer Vision

GitHub repository: https://github.com/batreller/CM4126-Coursework

Dataset: ISIC 2019 Challenge (25331 images, 8 categories)

Paper: Matsoukas, C. *et al.* (2022) 'What makes transfer learning work for medical images: feature reuse & other factors', *2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*. New Orleans (USA), 18-24 June. Piscataway, NJ: IEEE. pp. 9215-9224. Available at: https://doi.org/10.1109/CVPR52688.2022.00901 (Accessed: 13 November 2026).

...



# Research and Data Exploration
...




First let's create a function to download all datasets, it will only download datasets if they haven't been downloaded previously. It also extracts ZIP archives and it puts all downloaded data into `data/raw/isic2019` folder

In [1]:
import zipfile
from datetime import date
from pathlib import Path

import requests
from tqdm import tqdm

DATA_DIR = Path("data/raw/isic2019")
BASE_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/"
ISIC_FILES = [
    "ISIC_2019_Training_GroundTruth.csv",  # 1 mb
    "ISIC_2019_Training_Metadata.csv",  # 1 mb
    "ISIC_2019_Test_GroundTruth.csv",  # 454 kb
    "ISIC_2019_Test_Metadata.csv",  # 287 kb
    "ISIC_2019_Training_Input.zip",  # 9.1 gb
    "ISIC_2019_Test_Input.zip",  # 3.6 gb
]

# gets a remote file size
def remote_size(url):
    response = requests.head(url, timeout=30)
    response.raise_for_status()
    return int(response.headers["Content-Length"])

# downloads a file
def download_file(url, target):
    part = target.with_name(target.name + ".part")  # so that interrupted download didn't look like it was completed
    with requests.get(url, stream=True, timeout=30) as response:
        response.raise_for_status()
        total = int(response.headers["Content-Length"])
        with open(part, "wb") as out, tqdm(total=total, unit="B", unit_scale=True, desc=target.name) as bar:  # to visually display downloading progress
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                out.write(chunk)
                bar.update(len(chunk))
    part.replace(target)  # rename file at the end


def download_isic2019(data_dir=DATA_DIR, delete_zips=True):
    data_dir.mkdir(parents=True, exist_ok=True)

    for name in ISIC_FILES:
        url = BASE_URL + name
        target = data_dir / name

        # empty marker for zip files, as zip files are removed after extraction
        # this file is used to know whether zip was already downloaded and extracted or wasn't even downloaded yet
        extracted_marker = data_dir / (name + ".extracted")

        if extracted_marker.exists():
            print(f"{name}: already downloaded and extracted")
            continue

        if target.exists() and target.stat().st_size == remote_size(url):
            print(f"{name}: already downloaded")
        else:
            download_file(url, target)

        if name.endswith(".zip"):
            with zipfile.ZipFile(target) as archive:
                for member in tqdm(archive.infolist(), desc=f"Extracting {name}", unit="file"):
                    archive.extract(member, data_dir)
            extracted_marker.touch()  # create empty marker file only after extraction is finished
            if delete_zips:
                target.unlink()  #  remove zip

    stamp = data_dir / "DOWNLOADED_AT.txt"
    if not stamp.exists():
        stamp.write_text(date.today().isoformat())
    print("Dataset downloaded on:", stamp.read_text())

In [2]:
# download datasets
download_isic2019()

ISIC_2019_Training_GroundTruth.csv: already downloaded
ISIC_2019_Training_Metadata.csv: already downloaded
ISIC_2019_Test_GroundTruth.csv: already downloaded
ISIC_2019_Test_Metadata.csv: already downloaded
ISIC_2019_Training_Input.zip: already downloaded and extracted
ISIC_2019_Test_Input.zip: already downloaded and extracted
Dataset downloaded on: 2026-11-14


In [3]:
import pandas as pd

CLASS_NAMES = {
    "MEL": "Melanoma",  # malignant, the most dangerous skin cancer
    "NV": "Melanocytic nevus",  # benign
    "BCC": "Basal cell carcinoma",  # malignant, spreads very rarely
    "AK": "Actinic keratosis",  # pre-cancerous
    "BKL": "Benign keratosis",  # benign
    "DF": "Dermatofibroma",  # benign
    "VASC": "Vascular lesion",  # benign
    "SCC": "Squamous cell carcinoma",  # malignant
}
CLASSES = list(CLASS_NAMES)

# loads labels and metadata into one table with 1 row per image
def load_split(split):
    labels = pd.read_csv(DATA_DIR / f"ISIC_2019_{split}_GroundTruth.csv")
    metadata = pd.read_csv(DATA_DIR / f"ISIC_2019_{split}_Metadata.csv")

    # labels are encoded with one-hot so we convert each row into single class name
    # UNK must be included, otherwise all zero rows will be handled as the first class
    labels["label"] = labels[CLASSES + ["UNK"]].idxmax(axis=1)

    # take only `image` and `label` from `labels` and merge it with `metadata`.
    df = labels[["image", "label"]].merge(metadata, on="image", how="left", validate="one_to_one")
    # add `path` column with full path to the image
    df["path"] = [DATA_DIR / f"ISIC_2019_{split}_Input" / f"{name}.jpg" for name in df["image"]]
    return df

In [4]:
# now let's load both training and test splits
train_df = load_split("Training")
test_df = load_split("Test")

# and see their shapes
print("Train:", train_df.shape, " Test:", test_df.shape)
train_df.head()

Train: (25331, 7)  Test: (8238, 6)


,image,label,age_approx,anatom_site_general,lesion_id,sex,path
0,ISIC_0000000,NV,55.0,anterior torso,NaN,female,data\raw\isic2019\ISIC_2019_Training_Input\ISI...
1,ISIC_0000001,NV,30.0,anterior torso,NaN,female,data\raw\isic2019\ISIC_2019_Training_Input\ISI...
2,ISIC_0000002,MEL,60.0,upper extremity,NaN,female,data\raw\isic2019\ISIC_2019_Training_Input\ISI...
3,ISIC_0000003,NV,30.0,upper extremity,NaN,male,data\raw\isic2019\ISIC_2019_Training_Input\ISI...
4,ISIC_0000004,MEL,80.0,posterior torso,NaN,male,data\raw\isic2019\ISIC_2019_Training_Input\ISI...
